# **Protein mutation ML**

### **Single mutant fitness score (from Nisthal et al. 2019)**
proper dataset with embeddings

In [1]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import re
import mavenn

I0000 00:00:1779186113.683152   71417 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779186113.716078   71417 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779186114.280673   71417 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ============================================================
# LOAD DATASET
# ============================================================

data_df = mavenn.load_example_dataset('nisthal')

print("Original shape:", data_df.shape)
display(data_df.head())

Original shape: (813, 3)


,x,name,y
0,AYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...,T02A,0.4704
1,DYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...,T02D,0.5538
2,EYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...,T02E,-0.1299
3,FYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...,T02F,-0.3008
4,GYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...,T02G,0.6680


In [3]:
# ============================================================
# REMOVE EXTREME OUTLIERS
# ============================================================

# Keep only moderately stable/destabilized mutants
data_df = data_df[
    (data_df['y'] > -3) &
    (data_df['y'] < 3)
].copy()

print("After filtering:", data_df.shape)

After filtering: (801, 3)


In [4]:
# ============================================================
# EXTRACT MUTATION POSITION
# ============================================================

def get_position(name):
    return int(re.findall(r'\d+', name)[0])

data_df['pos'] = data_df['name'].apply(get_position)

print(data_df.columns)

Index(['x', 'name', 'y', 'pos'], dtype='str')


In [5]:
# =========================================================
# SELECT ONE MUTATION PER POSITION
# =========================================================

# Randomly select ONE mutant from each mutation position
position_representatives = (
    data_df
    .groupby('pos')
    .sample(n=1, random_state=42)
    .reset_index(drop=True)
)

print(position_representatives.columns)

print("\nUnique positions selected:")
print(position_representatives['pos'].nunique())

print("\nFrequency of positions:")
print(position_representatives['pos'].value_counts())

Index(['x', 'name', 'y', 'pos'], dtype='str')

Unique positions selected:
54

Frequency of positions:
pos
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
20    1
21    1
22    1
23    1
24    1
25    1
26    1
27    1
28    1
29    1
30    1
31    1
32    1
33    1
34    1
35    1
36    1
37    1
38    1
39    1
40    1
41    1
42    1
44    1
45    1
46    1
47    1
48    1
49    1
50    1
51    1
52    1
53    1
54    1
55    1
56    1
Name: count, dtype: int64


In [6]:
# ============================================================
# SAMPLE 50 UNIQUE POSITIONS
# ============================================================

subset_df = (
    position_representatives
    .sample(50, random_state=42)
    .reset_index(drop=True)
)

print("\nFinal subset shape:", subset_df.shape)

print("\nPosition frequencies:")
print(subset_df['pos'].value_counts())


Final subset shape: (50, 4)

Position frequencies:
pos
21    1
52    1
51    1
14    1
47    1
7     1
19    1
55    1
5     1
34    1
15    1
10    1
28    1
8     1
36    1
6     1
39    1
26    1
48    1
35    1
53    1
17    1
11    1
18    1
32    1
38    1
27    1
13    1
2     1
50    1
29    1
33    1
42    1
31    1
49    1
3     1
23    1
4     1
44    1
41    1
37    1
25    1
46    1
12    1
24    1
20    1
56    1
22    1
9     1
45    1
Name: count, dtype: int64


In [7]:
# ============================================================
# VERIFY ALL POSITIONS ARE UNIQUE
# ============================================================

assert subset_df['pos'].nunique() == 50

print("\n✅ All 50 mutations come from different positions.")


✅ All 50 mutations come from different positions.


In [8]:
# ============================================================
# SAVE DATASET IMMEDIATELY
# ============================================================

subset_df.to_csv(
    "selected_mutants.csv",
    index=False
)

print("\nDataset saved as:")
print("selected_mutants.csv")


Dataset saved as:
selected_mutants.csv


In [9]:
# ============================================================
# PREVIEW FINAL DATASET
# ============================================================

display(
    subset_df[
        ['name', 'pos', 'y', 'x']
    ].head()
)


,name,pos,y,x
0,V21P,21,-0.8258,TYKLILNGKTLKGETTTEAPDAATAEKVFKQYANDNGVDGEWTYDD...
1,F52M,52,2.7060,TYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...
2,T51M,51,0.9797,TYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...
3,G14L,14,-1.2500,TYKLILNGKTLKLETTTEAVDAATAEKVFKQYANDNGVDGEWTYDD...
4,D47Y,47,-0.3546,TYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDY...


In [10]:
sequences = subset_df['x'].tolist()

In [11]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: NVIDIA GeForce RTX 4070 Ti


In [12]:
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda")

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model = AutoModel.from_pretrained("facebook/esm2_t33_650M_UR50D")

model = model.to(device)
model.eval()
torch.set_grad_enabled(False)

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [13]:
print(type(model))
sequences = subset_df['x'].tolist()
len(sequences)

<class 'transformers.models.esm.modeling_esm.EsmModel'>


50

In [14]:
def get_embeddings_batch(sequences, batch_size=8):
    all_embeddings = []

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        hidden = outputs.last_hidden_state  # (B, L, d)

        # mean pooling
        embeddings = hidden.mean(dim=1)  # (B, d)

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

In [15]:
X = get_embeddings_batch(sequences, batch_size=8)
print(X.shape)

torch.Size([50, 1280])


In [16]:
X = X.numpy()
y = subset_df['y'].values

In [17]:
np.isnan(X).any()


np.False_

In [18]:
corr = np.corrcoef(X[:, 0], y)[0, 1]
print("Sample correlation:", corr)
print(X.shape)

Sample correlation: 0.327557389660366
(50, 1280)


In [19]:
subset_df['embedding'] = X.tolist()

In [20]:
subset_df.to_pickle("mutant_dataset.pkl")

### Automation process to generate 3D coordinates of the 50 selected sequences using FoldX